# Tugas 2 — Retrieval Data dan Pembuatan Mesin Pencari Sederhana
## Koleksi: Budaya Indonesia (Tradisi dan Kesenian) — Indonesia Kaya

Notebook ini menjalankan dan menjelaskan seluruh tahap Tugas 2 memakai source code di folder `src/`.
Dataset yang dipakai adalah hasil Tugas 1 (`data/dataset_crawling_ir.csv`), sehingga notebook dapat dijalankan **tanpa crawling ulang**.

**Alur sistem**

```
Koleksi Dokumen → Prapemrosesan → Indexing → Query Processing → Retrieval → Scoring → Ranking → Top-k → Antarmuka Mesin Pencari
```

| Tahap | File source code |
|---|---|
| 1. Prapemrosesan | `src/preprocessing.py` |
| 2. Inverted index | `src/indexing.py` |
| 3–4. TF-IDF, VSM, cosine similarity | `src/tfidf_retrieval.py` |
| 5. BM25 | `src/bm25.py` |
| Boolean retrieval | `src/boolean_retrieval.py` |
| 6. Mesin pencari | `src/search_engine.py` (CLI) dan `app.py` + `app_pages/` (dashboard Streamlit) |
| 7. Evaluasi | `src/evaluation.py`, ground truth di `evaluation/queries.csv` |

**Cara menjalankan notebook**

- **Di komputer:** clone repository (`git clone https://github.com/Marcelasetiawan/Tugas_Temu-Kembali-Informasi.git`)
  atau ekstrak ZIP project, pasang library (`python -m pip install -r requirements.txt`), buka notebook ini dari
  folder project, lalu **Run All**.
- **Di Google Colab:** buka notebook ini lewat *File → Open notebook → GitHub*, lalu **Runtime → Run all**.
  Sel persiapan Colab di bawah otomatis meng-clone repository dan memasang library.

## 0. Persiapan

In [1]:
# Persiapan khusus Google Colab: clone repository dan pasang library.
# Di komputer lokal sel ini dilewati otomatis (tidak melakukan apa-apa).
import os, sys

if "google.colab" in sys.modules and not os.path.exists("src"):
    !git clone -q https://github.com/Marcelasetiawan/Tugas_Temu-Kembali-Informasi.git
    %cd Tugas_Temu-Kembali-Informasi
    !pip install -q -r requirements.txt
else:
    print("Dijalankan di komputer lokal, folder project:", os.getcwd())

Dijalankan di komputer lokal, folder project: C:\Users\LENOVO\Documents\Temu Kembali Informasi\Tugas\tugas2_ir


In [2]:
import math
import pandas as pd

from src.preprocessing import Preprocessor, load_dataset, detect_proper_nouns, preprocess_corpus, STEM_OVERRIDES
from src.indexing import InvertedIndex
from src.tfidf_retrieval import TFIDFRetrieval
from src.bm25 import BM25
from src.boolean_retrieval import BooleanRetrieval
from src.search_engine import SearchEngine, DATA_PATH, PREPROCESSED_PATH, INDEX_PATH, STEM_CACHE_PATH
from src.evaluation import load_queries, evaluate, summarize

pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 200)
print("Library dan modul siap.")

Library dan modul siap.


## 1. Tahap 1 — Prapemrosesan

### 1.1 Memuat dataset, nilai kosong, dan dokumen duplikat

`load_dataset()` membaca CSV hasil Tugas 1, memeriksa kolom wajib (`doc_id, title, url, content, crawl_date`), lalu:
- mengisi `title` kosong dengan string kosong dan **membuang dokumen dengan `content` kosong** (tidak bisa dicari),
- **membuang duplikat** berdasarkan `doc_id`, `url`, dan `content`.

In [3]:
df, clean_report = load_dataset(DATA_PATH)

print("Nilai kosong per kolom wajib:", clean_report["Nilai kosong per kolom wajib"])
display(pd.DataFrame(
    [(k, v) for k, v in clean_report.items() if k != "Nilai kosong per kolom wajib"],
    columns=["Tahap", "Jumlah dokumen"],
))
display(df[["doc_id", "title", "url", "crawl_date"]].head())

Nilai kosong per kolom wajib: {'doc_id': 0, 'title': 0, 'url': 0, 'content': 0, 'crawl_date': 0}


,Tahap,Jumlah dokumen
0,Dokumen awal,516
1,Setelah buang content kosong,516
2,"Setelah buang duplikat (doc_id, url, content)",516


,doc_id,title,url,crawl_date
0,D1,Ngiling Bumbu: Tradisi Pemersatu Khas Warga Merangin Jambi,https://indonesiakaya.com/pustaka-indonesia/ngiling-bumbu-tradisi-pemersatu-khas-warga-merangin-jambi/,2026-10-14 01:24:14
1,D2,Muwon Namo: Ritual Memohon Hujan ala Suku Batin IX di Jambi,https://indonesiakaya.com/pustaka-indonesia/muwon-namo-ritual-memohon-hujan-ala-suku-batin-ix-di-jambi/,2026-10-14 01:24:16
2,D3,"Arakan Pompong, Tradisi Nelayan Suku Duano di Pesisir Jambi",https://indonesiakaya.com/pustaka-indonesia/arakan-pompong-tradisi-nelayan-suku-duano-di-pesisir-jambi/,2026-10-14 01:24:18
3,D4,"Mengungkap Makna Sakral di Balik Sayyang Pattuduq, Tradisi Kuda Penari Khas Tanah Mandar",https://indonesiakaya.com/pustaka-indonesia/sayyang-pattuduq/,2026-10-14 01:24:20
4,D5,Sandeq: Perahu Tercepat Warisan Suku Mandar di Nusantara,https://indonesiakaya.com/pustaka-indonesia/perahu-sandeq/,2026-10-14 01:24:22


### 1.2 Pipeline prapemrosesan

Teks yang diproses adalah **judul + isi** dokumen. Query pengguna diproses dengan pipeline yang **sama persis**.

| No | Tahap | Keterangan |
|---|---|---|
| 1 | Case folding | Semua huruf menjadi huruf kecil, agar *Tari* dan *tari* dianggap sama |
| 2 | Normalisasi | Menghapus aksen dan karakter non-ASCII, URL, apostrof (*Ma'nene → manene*), angka, tanda baca, dan simbol |
| 3 | Tokenisasi | Memecah teks menjadi kata berdasarkan spasi |
| 4 | Stopword removal | Membuang kata fungsi (stopword Sastrawi + tambahan) dan token < 2 huruf |
| 5 | Stemming | Mengubah kata ke bentuk dasar dengan **Sastrawi**; **nama diri tidak di-stem** |
| 6 | Stopword tahap 2 | Membuang kata dasar umum yang baru muncul setelah stemming (*menjadi → jadi*, *memiliki → milik*) |

**Alasan pemilihan stopword.** Daftar Sastrawi (126 kata) berisi kata fungsi bahasa Indonesia seperti *yang, dan, di, dengan, untuk* yang muncul di hampir semua dokumen dan tidak membedakan topik. Ditambah kata keterangan yang sering muncul di artikel (*juga, telah, tersebut, hingga, dapat*). Kata bertema budaya seperti *tradisi* atau *budaya* **tidak** dibuang walaupun sering muncul, karena bobotnya sudah dikecilkan otomatis oleh IDF.

**Mengapa nama diri tidak di-stem.** Sastrawi memotong akhiran *-i/-an/-kan* tanpa mengenali nama diri, sehingga *bali → bal*, *maluku → malu* (menjadi sama dengan kata "malu"), dan *minang → mang*. Nama diri dideteksi otomatis: kata yang di tengah kalimat diawali huruf kapital pada ≥ 90% kemunculannya (minimal 3 kali).

**Koreksi stemming.** Sastrawi juga membuang akhiran *-kah/-lah* karena mengira partikel (seperti *apakah*, *inilah*), padahal bagian dari kata dasar: *menikah → meni*, *melangkah → lang*. Kata yang terbukti salah di koleksi ini dikoreksi lewat daftar `STEM_OVERRIDES` (contoh: *menikah → nikah*), sehingga cocok dengan *pernikahan → nikah*.

In [4]:
proper_nouns = detect_proper_nouns(df["content"])
pre = Preprocessor(proper_nouns=proper_nouns, stem_cache_path=STEM_CACHE_PATH)
print("Jumlah stopword :", len(pre.stopwords))
print("Jumlah nama diri:", len(proper_nouns))

contoh = ["bali", "maluku", "kutai", "minang", "pekalongan", "toraja", "menikah", "melangkah"]
display(pd.DataFrame({
    "kata": contoh,
    "jika di-stem Sastrawi": [pre.stemmer.stem(w) for w in contoh],
    "nama diri / dikoreksi": [("nama diri" if w in proper_nouns else
                               "koreksi" if w in STEM_OVERRIDES else "-") for w in contoh],
    "hasil akhir": [pre.preprocess(w) for w in contoh],
}))

Jumlah stopword : 139
Jumlah nama diri: 1068


,kata,jika di-stem Sastrawi,nama diri / dikoreksi,hasil akhir
0,bali,bal,nama diri,[bali]
1,maluku,malu,nama diri,[maluku]
2,kutai,kuta,nama diri,[kutai]
3,minang,mang,nama diri,[minang]
4,pekalongan,kalong,nama diri,[pekalongan]
5,toraja,toraja,nama diri,[toraja]
6,menikah,meni,koreksi,[nikah]
7,melangkah,lang,koreksi,[langkah]


### 1.3 Contoh teks sebelum dan sesudah prapemrosesan

In [5]:
teks = df.loc[df["doc_id"] == "D30", "title"].iloc[0] + ". " + \
       " ".join(df.loc[df["doc_id"] == "D30", "content"].iloc[0].split()[:35])

steps = pre.steps(teks)
pd.set_option("display.max_colwidth", 500)
display(pd.DataFrame([(k, str(v)) for k, v in steps.items()], columns=["Tahap", "Hasil"]))
pd.set_option("display.max_colwidth", 150)
print(f"Jumlah token: {len(steps['3. Tokenisasi'])} -> {len(steps['6. Stopword tahap 2'])}")

,Tahap,Hasil
0,0. Teks asli,"Rambu Solo, Ritual Kematian Suku Toraja yang Sakral. Bagi kebanyakan masyarakat adat di Nusantara, siklus hidup manusia (mulai dari mengandung hingga kematian) merupakan suatu proses yang dianggap sakral. Karenanya, masyarakat adat di Nusantara masih menganggap hal-hal seperti ibu mengandung, melahirkan, masa kecil, dewasa,"
1,1. Case folding,"rambu solo, ritual kematian suku toraja yang sakral. bagi kebanyakan masyarakat adat di nusantara, siklus hidup manusia (mulai dari mengandung hingga kematian) merupakan suatu proses yang dianggap sakral. karenanya, masyarakat adat di nusantara masih menganggap hal-hal seperti ibu mengandung, melahirkan, masa kecil, dewasa,"
2,2. Normalisasi,rambu solo ritual kematian suku toraja yang sakral bagi kebanyakan masyarakat adat di nusantara siklus hidup manusia mulai dari mengandung hingga kematian merupakan suatu proses yang dianggap sakral karenanya masyarakat adat di nusantara masih menganggap hal hal seperti ibu mengandung melahirkan masa kecil dewasa
3,3. Tokenisasi,"['rambu', 'solo', 'ritual', 'kematian', 'suku', 'toraja', 'yang', 'sakral', 'bagi', 'kebanyakan', 'masyarakat', 'adat', 'di', 'nusantara', 'siklus', 'hidup', 'manusia', 'mulai', 'dari', 'mengandung', 'hingga', 'kematian', 'merupakan', 'suatu', 'proses', 'yang', 'dianggap', 'sakral', 'karenanya', 'masyarakat', 'adat', 'di', 'nusantara', 'masih', 'menganggap', 'hal', 'hal', 'seperti', 'ibu', 'mengandung', 'melahirkan', 'masa', 'kecil', 'dewasa']"
4,4. Stopword removal,"['rambu', 'solo', 'ritual', 'kematian', 'suku', 'toraja', 'sakral', 'kebanyakan', 'masyarakat', 'adat', 'nusantara', 'siklus', 'hidup', 'manusia', 'mulai', 'mengandung', 'kematian', 'merupakan', 'suatu', 'proses', 'dianggap', 'sakral', 'karenanya', 'masyarakat', 'adat', 'nusantara', 'menganggap', 'ibu', 'mengandung', 'melahirkan', 'masa', 'kecil', 'dewasa']"
5,5. Stemming,"['rambu', 'solo', 'ritual', 'mati', 'suku', 'toraja', 'sakral', 'banyak', 'masyarakat', 'adat', 'nusantara', 'siklus', 'hidup', 'manusia', 'mulai', 'kandung', 'mati', 'rupa', 'suatu', 'proses', 'anggap', 'sakral', 'karena', 'masyarakat', 'adat', 'nusantara', 'anggap', 'ibu', 'kandung', 'lahir', 'masa', 'kecil', 'dewasa']"
6,6. Stopword tahap 2,"['rambu', 'solo', 'ritual', 'mati', 'suku', 'toraja', 'sakral', 'masyarakat', 'adat', 'nusantara', 'siklus', 'hidup', 'manusia', 'mulai', 'kandung', 'mati', 'suatu', 'proses', 'anggap', 'sakral', 'masyarakat', 'adat', 'nusantara', 'anggap', 'ibu', 'kandung', 'lahir', 'masa', 'kecil', 'dewasa']"


Jumlah token: 44 -> 30


**Dampak terhadap retrieval** — contoh berikut menunjukkan bahwa variasi kata dengan imbuhan berbeda menjadi satu term yang sama, sehingga query *"tarian"* juga menemukan dokumen yang menulis *"tari"*, *"menari"*, atau *"penari"*:

In [6]:
variasi = ["tari", "tarian", "menari", "penari", "pemakaman", "dimakamkan", "makam",
           "pernikahan", "menikah", "kesenian", "seniman"]
display(pd.DataFrame({"kata asli": variasi, "term setelah prapemrosesan": [pre.preprocess(w) for w in variasi]}))

,kata asli,term setelah prapemrosesan
0,tari,[tari]
1,tarian,[tari]
2,menari,[tari]
3,penari,[tari]
4,pemakaman,[makam]
5,dimakamkan,[makam]
6,makam,[makam]
7,pernikahan,[nikah]
8,menikah,[nikah]
9,kesenian,[seni]


### 1.4 Prapemrosesan seluruh koleksi → `data/dataset_preprocessed.csv`

In [7]:
df = preprocess_corpus(df, pre, output_path=PREPROCESSED_PATH)

print("Total token seluruh koleksi :", df["jumlah_token"].sum())
print("Rata-rata token per dokumen :", round(df["jumlah_token"].mean(), 1))
print("Disimpan ke", PREPROCESSED_PATH.relative_to(PREPROCESSED_PATH.parents[1]))
display(df[["doc_id", "title", "jumlah_kata", "jumlah_token"]].head())

Total token seluruh koleksi : 193833
Rata-rata token per dokumen : 375.6
Disimpan ke data\dataset_preprocessed.csv


,doc_id,title,jumlah_kata,jumlah_token
0,D1,Ngiling Bumbu: Tradisi Pemersatu Khas Warga Merangin Jambi,859,607
1,D2,Muwon Namo: Ritual Memohon Hujan ala Suku Batin IX di Jambi,595,448
2,D3,"Arakan Pompong, Tradisi Nelayan Suku Duano di Pesisir Jambi",960,666
3,D4,"Mengungkap Makna Sakral di Balik Sayyang Pattuduq, Tradisi Kuda Penari Khas Tanah Mandar",847,635
4,D5,Sandeq: Perahu Tercepat Warisan Suku Mandar di Nusantara,624,430


## 2. Tahap 2 — Inverted Index

Struktur dasar: **term → postings list**. Setiap posting menyimpan ID dokumen dan **TF** (jumlah kemunculan term di dokumen tersebut). **DF** adalah jumlah dokumen di postings list.

```
index["toraja"] = {"D22": 14, "D23": 16, "D30": 12, ...}
```

Index disimpan ke `index/inverted_index.json`.

In [8]:
index = InvertedIndex().build(dict(zip(df["doc_id"], df["tokens"])))
index.save(INDEX_PATH)

print("Jumlah dokumen (N)        :", index.N)
print("Jumlah term (vocabulary)  :", len(index.vocabulary))
print("Rata-rata panjang (avgdl) :", round(index.avgdl, 2), "token")

Jumlah dokumen (N)        : 516
Jumlah term (vocabulary)  : 11639
Rata-rata panjang (avgdl) : 375.65 token


### 2.1 Contoh dictionary dan postings list

In [9]:
for term in ["angklung", "calung", "toraja", "reog", "sasando"]:
    print(f"{term:9s} -> {index.postings(term)}")

angklung  -> ['D108', 'D110', 'D112', 'D255', 'D274', 'D280', 'D288', 'D289', 'D300', 'D343', 'D376', 'D403', 'D433', 'D457', 'D458', 'D501']
calung    -> ['D180', 'D188', 'D274', 'D276', 'D280', 'D300', 'D330', 'D375', 'D418', 'D501']
toraja    -> ['D22', 'D23', 'D30', 'D43', 'D130', 'D177', 'D421', 'D481']
reog      -> ['D28', 'D255', 'D343', 'D357', 'D376', 'D432', 'D433', 'D449', 'D461', 'D504']
sasando   -> ['D476']


### 2.2 Contoh TF dan DF

In [10]:
rows = []
for term in ["toraja", "makam", "ritual", "angklung", "budaya"]:
    postings = index.index[term]
    top = sorted(postings.items(), key=lambda x: x[1], reverse=True)[:3]
    rows.append({"term": term, "DF": index.df(term),
                 "TF tertinggi (doc_id: TF)": ", ".join(f"{d}: {tf}" for d, tf in top)})
display(pd.DataFrame(rows))

print("Term dengan DF tertinggi:")
display(pd.DataFrame(sorted(((t, index.df(t)) for t in index.vocabulary), key=lambda x: -x[1])[:10],
                     columns=["term", "DF"]))

,term,DF,TF tertinggi (doc_id: TF)
0,toraja,8,"D177: 17, D23: 16, D22: 14"
1,makam,17,"D22: 19, D107: 12, D23: 7"
2,ritual,142,"D96: 30, D2: 15, D86: 14"
3,angklung,16,"D403: 28, D274: 14, D289: 10"
4,budaya,333,"D243: 23, D142: 19, D199: 15"


Term dengan DF tertinggi:


,term,DF
0,masyarakat,391
1,orang,369
2,buat,348
3,budaya,333
4,buah,308
5,hidup,301
6,bentuk,299
7,nama,271
8,asal,269
9,hasil,265


## 3. Tahap 3 — TF-IDF dan Vector Space Model

**Rumus yang digunakan** (sesuai materi kuliah):

| Komponen | Rumus |
|---|---|
| TF | $TF(t,d)$ = jumlah kemunculan term $t$ di dokumen $d$ |
| IDF | $IDF(t) = \log_{10}\dfrac{N}{df(t)}$ |
| TF-IDF | $w_{t,d} = TF(t,d) \times IDF(t)$ |

Query juga diberi bobot TF-IDF: $w_{t,q} = TF(t,q) \times IDF(t)$.

**Verifikasi dengan latihan di materi kuliah** — N = 100, df(wifi) = 10, TF(wifi, D1) = 3:

In [11]:
N_, df_, tf_ = 100, 10, 3
idf_ = math.log10(N_ / df_)
print(f"IDF(wifi)        = log10({N_}/{df_}) = {idf_:.4f}")
print(f"TF-IDF(wifi, D1) = {tf_} x {idf_:.4f} = {tf_ * idf_:.4f}")
print(f"Jika df = 80     : IDF = {math.log10(100/80):.4f}, TF-IDF = {3*math.log10(100/80):.4f}  (term umum -> bobot kecil)")

IDF(wifi)        = log10(100/10) = 1.0000
TF-IDF(wifi, D1) = 3 x 1.0000 = 3.0000
Jika df = 80     : IDF = 0.0969, TF-IDF = 0.2907  (term umum -> bobot kecil)


### 3.1 Contoh perhitungan TF-IDF pada koleksi

In [12]:
tfidf = TFIDFRetrieval(index)

query_contoh = "ritual pemakaman suku toraja"
q_terms = pre.preprocess(query_contoh)
docs_contoh = ["D30", "D23", "D421", "D2"]
print("Query:", query_contoh, "->", q_terms, "| N =", index.N)
display(pd.DataFrame(tfidf.explain(q_terms, docs_contoh)))

Query: ritual pemakaman suku toraja -> ['ritual', 'makam', 'suku', 'toraja'] | N = 516


,term,doc_id,TF,DF,IDF = log10(N/DF),TF-IDF = TF x IDF
0,ritual,D30,2,142,0.5604,1.1207
1,ritual,D23,3,142,0.5604,1.6811
2,ritual,D421,0,142,0.5604,0.0000
3,ritual,D2,15,142,0.5604,8.4054
4,makam,D30,4,17,1.4822,5.9288
5,makam,D23,7,17,1.4822,10.3754
6,makam,D421,0,17,1.4822,0.0000
7,makam,D2,1,17,1.4822,1.4822
8,suku,D30,9,132,0.5921,5.3287
9,suku,D23,2,132,0.5921,1.1842


### 3.2 Representasi vektor dokumen dan query

Setiap dimensi vektor mewakili satu term di vocabulary (11.000+ dimensi). Tabel berikut menampilkan dimensi yang sesuai dengan term query saja; norma |d| dihitung dari **seluruh** term dokumen.

In [13]:
q_vec = tfidf.query_vector(q_terms)
vec_rows = {"Query": [round(q_vec.get(t, 0), 4) for t in q_terms]}
for d in docs_contoh:
    vec_rows[d] = [round(tfidf.doc_vectors[d].get(t, 0), 4) for t in q_terms]
vectors = pd.DataFrame(vec_rows, index=q_terms).T
vectors["|vektor| (semua term)"] = [round(math.sqrt(sum(w*w for w in q_vec.values())), 4)] + \
                                    [round(tfidf.doc_norm[d], 4) for d in docs_contoh]
vectors["jumlah term unik"] = [len(q_vec)] + [len(tfidf.doc_vectors[d]) for d in docs_contoh]
display(vectors)

,ritual,makam,suku,toraja,|vektor| (semua term),jumlah term unik
Query,0.5604,1.4822,0.5921,1.8096,2.4771,4
D30,1.1207,5.9288,5.3287,21.7147,56.7781,219
D23,1.6811,10.3754,1.1842,28.9530,74.0360,294
D421,0.0000,0.0000,0.5921,16.2860,75.1238,199
D2,8.4054,1.4822,4.1445,0.0000,73.0150,218


## 4. Tahap 4 — Cosine Similarity dan Ranking

$$\cos(q, d) = \frac{\vec{q} \cdot \vec{d}}{\lVert\vec{q}\rVert \times \lVert\vec{d}\rVert}$$

Cosine hanya dihitung untuk **dokumen kandidat**, yaitu dokumen yang muncul di postings list minimal satu term query.

### 4.1 Contoh perhitungan cosine satu dokumen

In [14]:
d = "D30"
dot = sum(w * tfidf.doc_vectors[d].get(t, 0) for t, w in q_vec.items())
q_norm = math.sqrt(sum(w * w for w in q_vec.values()))
print(f"q . d   = {dot:.4f}")
print(f"|q|     = {q_norm:.4f}")
print(f"|d|     = {tfidf.doc_norm[d]:.4f}")
print(f"cos(q,d)= {dot:.4f} / ({q_norm:.4f} x {tfidf.doc_norm[d]:.4f}) = {dot / (q_norm * tfidf.doc_norm[d]):.4f}")
print(f"Hasil fungsi tfidf.cosine(): {tfidf.cosine(q_vec, d):.4f}")

q . d   = 51.8647
|q|     = 2.4771
|d|     = 56.7781
cos(q,d)= 51.8647 / (2.4771 x 56.7781) = 0.3688
Hasil fungsi tfidf.cosine(): 0.3688


### 4.2 Top-10 hasil TF-IDF + cosine

In [15]:
engine = SearchEngine(verbose=False)

def tampilkan(query, method="bm25", top_k=10):
    res = engine.search(query, method=method, top_k=top_k)
    print(f"Query: '{query}' | Metode: {res['method']} | Term: {res['terms']} | Kandidat: {res['n_candidates']}")
    if res["message"]:
        print(">>", res["message"])
        return
    display(pd.DataFrame(res["results"])[["rank", "title", "url", "score", "snippet"]]
            .rename(columns={"rank": "peringkat", "title": "judul", "score": "skor"})
            .round({"skor": 4}))

tampilkan("ritual pemakaman suku toraja", method="tfidf")

Query: 'ritual pemakaman suku toraja' | Metode: TF-IDF + Cosine | Term: ['ritual', 'makam', 'suku', 'toraja'] | Kandidat: 240


,peringkat,judul,url,skor,snippet
0,1,Menyaksikan Kemegahan Pemakaman Tebing Khas Toraja di Londa dan Lemo,https://indonesiakaya.com/pustaka-indonesia/tradisi-menyimpan-mayat-londa-dan-lemo/,0.3906,"Londa adalah kawasan pemakaman Suku Toraja yang terdiri dari kuburan batu tempat menyimpan mayat, khusus untuk leluhur Toraja dan keturunannya."
1,2,Passiliran: Pemakaman Bayi di Dalam Pohon Khas Tana Toraja,https://indonesiakaya.com/pustaka-indonesia/desa-adat-toraja-sillanan/,0.3826,"Di dalam tubuh pohon inilah bayi-bayi dari Desa Kambira, Tana Toraja, dimakamkan."
2,3,"Menyusuri Bukit Batu Lemo, Kuburan Alam Suku Toraja",https://indonesiakaya.com/pustaka-indonesia/makam-tebing/,0.3785,Sementara Hetty Nooy-Palm dalam The Sa’dan-Toraja: A Study of Their Social Life and Religion menyebut tau-tau dibuat sebelum tahap kedua dalam rit...
3,4,"Rambu Solo, Ritual Kematian Suku Toraja yang Sakral",https://indonesiakaya.com/pustaka-indonesia/rambu-solo-ritual-kematian-suku-toraja-yang-sakral/,0.3688,"Dalam tradisi Suku Toraja, makam dalam peti justru dimasukkan ke dalam gua atau rumah kecil yang khusus menyimpan beberapa peti jenazah."
4,5,"Ma’nene, Cara Orang Toraja Mengenang Leluhurnya",https://indonesiakaya.com/pustaka-indonesia/manene-sebuah-prosesi-adat-bentuk-penghormatan-terhadap-para-leluhur/,0.3397,"Menurut Micha, orang-orang di Toraja Selatan memiliki ritual serupa untuk menghormati jenazah leluhurnya."
5,6,"Ngembang, Bentuk Penghormatan kepada Leluhur",https://indonesiakaya.com/pustaka-indonesia/ngembang-bentuk-penghormatan-kepada-leluhur/,0.2923,Namun terik matahari yang menyengat tidak menyulutkan niat para kokolot Kampung Budaya Sindang Barang dan beberapa warga yang ingin melakukan ritu...
6,7,"Paramba, Kain Khas Toraja dengan Warna-Warna Cerah",https://indonesiakaya.com/pustaka-indonesia/paramba-kain-khas-toraja-dengan-warna-warna-cerah/,0.1972,"Kain ini menjadi salah satu kain khas Desa Sad’an, Malimbong Tobarana, Kabupaten Toraja Utara."
7,8,Tari Batingna Lebonna dan Legenda Asal-Usul dari Tana Toraja,https://indonesiakaya.com/pustaka-indonesia/tari-batingna-lebonna-dan-legenda-asal-usul-dari-tana-toraja/,0.1603,"Suku Toraja terkenal dengan adat istiadatnya yang kental dan kesenian tari yang khas, baik dari gerak tari, musik, dan busananya."
8,9,"Seren Taun, Tradisi Panen Raya Penuh Makna di Sindang Barang",https://indonesiakaya.com/pustaka-indonesia/seren-taun-tradisi-panen-raya-penuh-makna-di-sindang-barang/,0.0974,"Esok hari setelah malam-nya dilakukan ritual neteupken, para kokolot melakukan ritual ngembang ke makam leluhur."
9,10,Menggali Kekayaan Tradisi Suku Naulu di Era Modern,https://indonesiakaya.com/pustaka-indonesia/tradisi-suku-naulu-dan-modernisasi-masa-kini/,0.0786,Salah satu kekayaan lain tersebut adalah kebudayaan unik dari suku-suku terasing di wilayah Maluku Tengah.


## 5. Tahap 5 — BM25

$$\text{score}(D,Q) = \sum_{t \in Q} IDF(t) \cdot \frac{f(t,D)\,(k_1+1)}{f(t,D) + k_1\left(1 - b + b\,\frac{|D|}{avgdl}\right)}
\qquad IDF(t) = \ln\left(1 + \frac{N - df(t) + 0.5}{df(t) + 0.5}\right)$$

Parameter: **k1 = 1,2** (saturasi TF) dan **b = 0,75** (normalisasi panjang dokumen), sesuai nilai awal yang disarankan.

### 5.1 Verifikasi dengan contoh di materi kuliah

Koleksi D1 = "hotel murah nyaman", D2 = "hotel murah dekat kampus", D3 = "kampus nyaman", query "hotel nyaman".

In [16]:
toy = InvertedIndex().build({"D1": "hotel murah nyaman".split(),
                             "D2": "hotel murah dekat kampus".split(),
                             "D3": "kampus nyaman".split()})
toy_bm25 = BM25(toy, k1=1.2, b=0.75)
print("avgdl =", toy.avgdl, "| IDF(hotel) =", round(toy_bm25.idf["hotel"], 4))
display(pd.DataFrame(toy_bm25.search(["hotel", "nyaman"], top_k=None), columns=["doc_id", "skor BM25"]).round(4))

# perhitungan manual D3 (|D| = 2): hanya term "nyaman" yang muncul
idf_n = math.log(1 + (3 - 2 + 0.5) / (2 + 0.5))
manual_d3 = idf_n * 1 * 2.2 / (1 + 1.2 * (1 - 0.75 + 0.75 * 2 / 3))
print(f"Manual D3 = {idf_n:.4f} x 2.2 / (1 + 1.2 x (0.25 + 0.75 x 2/3)) = {manual_d3:.4f}")

avgdl = 3.0 | IDF(hotel) = 0.47


,doc_id,skor BM25
0,D1,0.9400
1,D3,0.5442
2,D2,0.4136


Manual D3 = 0.4700 x 2.2 / (1 + 1.2 x (0.25 + 0.75 x 2/3)) = 0.5442


Skor **D1 = 0,9400** dan urutan **D1 → D3 → D2** sama persis dengan materi kuliah. Skor D3 dan D2 sedikit berbeda dari slide (0,5313 dan 0,4043); perhitungan manual di atas dengan rumus yang sama menghasilkan 0,5442, sehingga perbedaan kemungkinan berasal dari pembulatan/perhitungan di slide.

### 5.2 Contoh perhitungan BM25 pada koleksi

In [17]:
bm25 = BM25(index, k1=1.2, b=0.75)
print("Query:", q_terms, "| N =", index.N, "| avgdl =", round(index.avgdl, 2))
display(pd.DataFrame(bm25.explain(q_terms, ["D30", "D23", "D421"])))

Query: ['ritual', 'makam', 'suku', 'toraja'] | N = 516 | avgdl = 375.65


,term,doc_id,"f(t,D)",DF,|D|,|D|/avgdl,IDF,skor term
0,ritual,D30,2,142,455,1.211,1.2887,1.6726
1,ritual,D23,3,142,599,1.595,1.2887,1.7962
2,ritual,D421,0,142,388,1.033,1.2887,0.0000
3,makam,D30,4,17,455,1.211,3.3858,5.5278
4,makam,D23,7,17,599,1.595,3.3858,5.9692
5,makam,D421,0,17,388,1.033,3.3858,0.0000
6,suku,D30,9,132,455,1.211,1.3615,2.5945
7,suku,D23,2,132,599,1.595,1.3615,1.6038
8,suku,D421,1,132,388,1.033,1.3615,1.3434
9,toraja,D30,12,8,455,1.211,4.1080,8.0993


### 5.3 Top-10 hasil BM25

In [18]:
tampilkan("ritual pemakaman suku toraja", method="bm25")

Query: 'ritual pemakaman suku toraja' | Metode: BM25 | Term: ['ritual', 'makam', 'suku', 'toraja'] | Kandidat: 240


,peringkat,judul,url,skor,snippet
0,1,"Rambu Solo, Ritual Kematian Suku Toraja yang Sakral",https://indonesiakaya.com/pustaka-indonesia/rambu-solo-ritual-kematian-suku-toraja-yang-sakral/,17.8941,"Dalam tradisi Suku Toraja, makam dalam peti justru dimasukkan ke dalam gua atau rumah kecil yang khusus menyimpan beberapa peti jenazah."
1,2,"Menyusuri Bukit Batu Lemo, Kuburan Alam Suku Toraja",https://indonesiakaya.com/pustaka-indonesia/makam-tebing/,17.5226,Sementara Hetty Nooy-Palm dalam The Sa’dan-Toraja: A Study of Their Social Life and Religion menyebut tau-tau dibuat sebelum tahap kedua dalam rit...
2,3,Menyaksikan Kemegahan Pemakaman Tebing Khas Toraja di Londa dan Lemo,https://indonesiakaya.com/pustaka-indonesia/tradisi-menyimpan-mayat-londa-dan-lemo/,17.2317,"Londa adalah kawasan pemakaman Suku Toraja yang terdiri dari kuburan batu tempat menyimpan mayat, khusus untuk leluhur Toraja dan keturunannya."
3,4,Passiliran: Pemakaman Bayi di Dalam Pohon Khas Tana Toraja,https://indonesiakaya.com/pustaka-indonesia/desa-adat-toraja-sillanan/,15.9649,"Di dalam tubuh pohon inilah bayi-bayi dari Desa Kambira, Tana Toraja, dimakamkan."
4,5,"Ma’nene, Cara Orang Toraja Mengenang Leluhurnya",https://indonesiakaya.com/pustaka-indonesia/manene-sebuah-prosesi-adat-bentuk-penghormatan-terhadap-para-leluhur/,13.2532,"Menurut Micha, orang-orang di Toraja Selatan memiliki ritual serupa untuk menghormati jenazah leluhurnya."
5,6,Tari Batingna Lebonna dan Legenda Asal-Usul dari Tana Toraja,https://indonesiakaya.com/pustaka-indonesia/tari-batingna-lebonna-dan-legenda-asal-usul-dari-tana-toraja/,9.2946,"Suku Toraja terkenal dengan adat istiadatnya yang kental dan kesenian tari yang khas, baik dari gerak tari, musik, dan busananya."
6,7,"Ngembang, Bentuk Penghormatan kepada Leluhur",https://indonesiakaya.com/pustaka-indonesia/ngembang-bentuk-penghormatan-kepada-leluhur/,9.2722,Namun terik matahari yang menyengat tidak menyulutkan niat para kokolot Kampung Budaya Sindang Barang dan beberapa warga yang ingin melakukan ritu...
7,8,Muwon Namo: Ritual Memohon Hujan ala Suku Batin IX di Jambi,https://indonesiakaya.com/pustaka-indonesia/muwon-namo-ritual-memohon-hujan-ala-suku-batin-ix-di-jambi/,8.2398,Ritual memohon hujan ini telah lama menjadi bagian dari kehidupan masyarakat suku Batin Sembilan.
8,9,"Seren Taun, Tradisi Panen Raya Penuh Makna di Sindang Barang",https://indonesiakaya.com/pustaka-indonesia/seren-taun-tradisi-panen-raya-penuh-makna-di-sindang-barang/,7.9108,"Esok hari setelah malam-nya dilakukan ritual neteupken, para kokolot melakukan ritual ngembang ke makam leluhur."
9,10,"Paramba, Kain Khas Toraja dengan Warna-Warna Cerah",https://indonesiakaya.com/pustaka-indonesia/paramba-kain-khas-toraja-dengan-warna-warna-cerah/,7.6186,"Kain ini menjadi salah satu kain khas Desa Sad’an, Malimbong Tobarana, Kabupaten Toraja Utara."


### 5.4 Analisis pengaruh TF, IDF, dan panjang dokumen

**(a) Pengaruh TF dan TF saturation.** Kontribusi satu term ($IDF = 1$, dokumen sepanjang avgdl) ketika TF naik. Pada TF-IDF bobot naik **linear**, sedangkan pada BM25 kenaikannya makin kecil dan mendekati batas $k_1 + 1 = 2{,}2$.

In [19]:
rows = []
for tf in [1, 2, 3, 5, 10, 20, 50]:
    bm = tf * (1.2 + 1) / (tf + 1.2 * (1 - 0.75 + 0.75 * 1.0))
    rows.append({"TF": tf, "TF-IDF (TF x IDF)": tf * 1.0, "BM25 (k1=1.2)": round(bm, 4),
                 "kenaikan BM25 dari baris sebelumnya": None})
sat = pd.DataFrame(rows)
sat["kenaikan BM25 dari baris sebelumnya"] = sat["BM25 (k1=1.2)"].diff().round(4)
display(sat)

,TF,TF-IDF (TF x IDF),BM25 (k1=1.2),kenaikan BM25 dari baris sebelumnya
0,1,1.0,1.0000,NaN
1,2,2.0,1.3750,0.3750
2,3,3.0,1.5714,0.1964
3,5,5.0,1.7742,0.2028
4,10,10.0,1.9643,0.1901
5,20,20.0,2.0755,0.1112
6,50,50.0,2.1484,0.0729


**(b) Pengaruh IDF terhadap term yang jarang muncul.** Term langka (DF kecil) mendapat IDF besar sehingga lebih menentukan skor.

In [20]:
terms_idf = ["budaya", "tradisi", "tari", "ritual", "toraja", "angklung", "sasando"]
display(pd.DataFrame([{
    "term": t, "DF": index.df(t),
    "IDF TF-IDF = log10(N/df)": round(tfidf.idf[t], 4),
    "IDF BM25 = ln(1+(N-df+0.5)/(df+0.5))": round(bm25.idf[t], 4),
} for t in terms_idf]))

,term,DF,IDF TF-IDF = log10(N/df),IDF BM25 = ln(1+(N-df+0.5)/(df+0.5))
0,budaya,333,0.1902,0.4384
1,tradisi,244,0.3253,0.7488
2,tari,217,0.3762,0.8658
3,ritual,142,0.5604,1.2887
4,toraja,8,1.8096,4.1080
5,angklung,16,1.5085,3.4447
6,sasando,1,2.7126,5.8426


**(c) Pengaruh panjang dokumen.** Skor satu term dengan TF = 3 pada dokumen yang panjangnya berbeda. Dengan b = 0,75 dokumen panjang dinormalisasi (skor turun); dengan b = 0 panjang dokumen diabaikan.

In [21]:
rows = []
for ratio in [0.25, 0.5, 1.0, 2.0, 4.0]:
    row = {"|D| / avgdl": ratio}
    for b in [0.0, 0.75, 1.0]:
        row[f"BM25 b={b}"] = round(3 * 2.2 / (3 + 1.2 * (1 - b + b * ratio)), 4)
    rows.append(row)
display(pd.DataFrame(rows))

,|D| / avgdl,BM25 b=0.0,BM25 b=0.75,BM25 b=1.0
0,0.25,1.5714,1.8723,2.0000
1,0.50,1.5714,1.7600,1.8333
2,1.00,1.5714,1.5714,1.5714
3,2.00,1.5714,1.2941,1.2222
4,4.00,1.5714,0.9565,0.8462


## 6. Tahap 6 — Mesin Pencari Sederhana

Kelas `SearchEngine` (`src/search_engine.py`) menggabungkan semua komponen:

1. **Input query** pengguna
2. **Prapemrosesan query** dengan `Preprocessor` yang sama dengan dokumen
3. **Retrieval kandidat** dari inverted index (gabungan postings list term query)
4. **Scoring** dengan TF-IDF + cosine atau BM25
5. **Ranking** dari skor terbesar dan pengambilan **Top-k**
6. **Tampilan** peringkat, judul, URL, skor, dan snippet (kalimat yang paling banyak memuat term query)
7. **Pesan** jika query kosong, hanya stopword, atau tidak ada dokumen yang cocok

Antarmuka tersedia dalam dua bentuk yang sama-sama bisa dipakai:

| Antarmuka | Perintah | Keterangan |
|---|---|---|
| Baris perintah (CLI) | `python -m src.search_engine` | Daftar contoh query (ketik nomornya untuk langsung mencari), menu metode bernomor (1. BM25, 2. TF-IDF + Cosine), pilihan Top-k, hasil berupa tabel ringkasan + detail. Ketik `contoh` untuk melihat contoh lagi, `keluar` untuk berhenti. |
| Dashboard web | `streamlit run app.py` | 9 menu: Dashboard, Panduan pemakaian, Cari dokumen, Bandingkan metode, Boolean retrieval, Inverted index, Evaluasi, Koleksi dokumen, Tentang sistem. |

Sel berikut memakai fungsi `print_result` dari `src/search_engine.py`, sehingga hasilnya tampil **sama seperti versi CLI**:

In [22]:
from src.search_engine import print_result

print_result(engine.search("alat musik bambu dari sunda", method="bm25", top_k=5))


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 HASIL PENCARIAN  "alat musik bambu dari sunda"
 Metode   : BM25
 Term     : alat, musik, bambu, sunda
 Kandidat : 300 dokumen memuat minimal satu term query
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 Top-5 dokumen (diurutkan dari skor tertinggi)
┌─────┬────────┬──────────────────────────────────────────────────────────┬──────────┐
│  No │ Doc ID │ Judul                                                    │     Skor │
├─────┼────────┼──────────────────────────────────────────────────────────┼──────────┤
│   1 │ D280   │ Calung: Merdunya Ketukan Bambu dari Tanah Sunda          │  12.0621 │
│   2 │ D403   │ Angklung: Warisan Budaya Sunda yang Diakui UNESCO        │  11.4685 │
│   3 │ D274   │ Angklung: Dari Tradisi Agraris Jawa Barat hingga...      │  11.4481 │
│   4 │ D418   │ Sejarah dan Ragam Calung, Ikon Budaya Sunda              │  11.3336 │
│   5 │ 

In [23]:
print_result(engine.search("kain tenun tradisional", method="tfidf", top_k=5))


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 HASIL PENCARIAN  "kain tenun tradisional"
 Metode   : TF-IDF + Cosine
 Term     : kain, tenun, tradisional
 Kandidat : 313 dokumen memuat minimal satu term query
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 Top-5 dokumen (diurutkan dari skor tertinggi)
┌─────┬────────┬──────────────────────────────────────────────────────────┬──────────┐
│  No │ Doc ID │ Judul                                                    │     Skor │
├─────┼────────┼──────────────────────────────────────────────────────────┼──────────┤
│   1 │ D514   │ Mengenal Lebih Dekat Kain Tenun Khas Ternate             │   0.7824 │
│   2 │ D258   │ Kain Tenun, Warisan Kekayaan Budaya Maluku               │   0.6850 │
│   3 │ D124   │ Tradisi Menenun, Peninggalan Nenek Moyang Suku Baduy     │   0.5832 │
│   4 │ D493   │ Tenun Sambas: Kain Tradisional Kalimantan Barat yang...  │   0.5535 │
│  

Contoh query yang **tidak menemukan dokumen**:

In [24]:
print_result(engine.search("smartphone android"))   # term tidak ada di koleksi
print_result(engine.search("yang dan di"))          # hanya stopword


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 HASIL PENCARIAN  "smartphone android"
 Metode   : BM25
 Term     : smartphone, android
 Kandidat : 0 dokumen memuat minimal satu term query
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 ✗ Tidak ditemukan dokumen untuk query 'smartphone android'. Term ['smartphone', 'android'] tidak ada di koleksi.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 HASIL PENCARIAN  "yang dan di"
 Metode   : BM25
 Term     : -
 Kandidat : 0 dokumen memuat minimal satu term query
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 ✗ Query hanya berisi stopword/simbol, tidak ada term yang bisa dicari.


Untuk mencoba query sendiri di notebook, hapus tanda `#` lalu jalankan sel berikut. Untuk pencarian berulang dengan menu lengkap, gunakan versi CLI (`python -m src.search_engine`) di terminal.

In [25]:
# query = input("Masukkan query: ")
# print_result(engine.search(query, method="bm25", top_k=10))

### 6.1 Boolean retrieval (pelengkap)

Boolean retrieval (exact match, tanpa skor) juga tersedia di `src/boolean_retrieval.py`, dengan operator `AND`, `OR`, `NOT`, dan tanda kurung.

In [26]:
for q in ["toraja AND (ritual OR makam)", "wayang AND (kulit OR golek)", "senjata AND dayak AND NOT tari"]:
    hasil = engine.boolean.search(q)
    print(f"{q:35s} -> {len(hasil)} dokumen: {hasil[:8]}")

toraja AND (ritual OR makam)        -> 5 dokumen: ['D22', 'D23', 'D30', 'D43', 'D177']
wayang AND (kulit OR golek)         -> 19 dokumen: ['D14', 'D47', 'D133', 'D143', 'D222', 'D239', 'D281', 'D295']
senjata AND dayak AND NOT tari      -> 7 dokumen: ['D61', 'D66', 'D101', 'D116', 'D122', 'D350', 'D451']


## 7. Tahap 7 — Evaluasi Retrieval

### 7.1 Query pengujian dan ground truth

Ground truth disusun **manual** dengan kriteria domain yang tertulis untuk setiap query (file `evaluation/queries.csv`): sebuah dokumen relevan jika **topik utamanya** sesuai kebutuhan informasi query.

Penyusunan dilakukan dua tahap:
1. **Penelusuran judul** — dokumen yang judulnya jelas membahas topik query.
2. **Pooling** — seluruh dokumen di Top-10 gabungan TF-IDF dan BM25 yang belum tercatat diperiksa isinya satu per satu (98 dokumen). Dari tahap ini ditemukan 9 dokumen relevan tambahan yang judulnya tidak menyebut kata kunci, misalnya *Tari Cenderawasih* (tarian Bali) untuk Q7 dan *Pedang Bara Sangihe* untuk Q10.

Ada **18 query**: 12 query umum (Q1–Q12) dan 6 query **pendek & spesifik** (Q13–Q18) untuk Eksperimen 3.

In [27]:
queries = load_queries("evaluation/queries.csv")
display(queries[["qid", "query", "jenis", "kriteria_relevansi"]].assign(
    jumlah_relevan=queries["relevant"].map(len)))

,qid,query,jenis,kriteria_relevansi,jumlah_relevan
0,Q1,ritual pemakaman suku toraja,umum,Artikel yang membahas ritual kematian/pemakaman atau penghormatan jenazah leluhur suku Toraja,5
1,Q2,rumah adat tradisional,umum,Artikel yang topik utamanya sebuah rumah adat/rumah tradisional suatu daerah,17
2,Q3,kain batik dan motifnya,umum,Artikel yang topik utamanya batik (kain/motif/kerajinan batik); tarian bernama Batik tidak termasuk,12
3,Q4,pertunjukan wayang,umum,Artikel yang topik utamanya wayang (jenis wayang atau pementasannya),9
4,Q5,cerita rakyat papua,umum,Artikel cerita rakyat/legenda yang berasal dari Papua,3
5,Q6,kain tenun tradisional,umum,Artikel yang topik utamanya kain tenun/songket/kain tradisional hasil tenun,17
6,Q7,tari tradisional bali,umum,Artikel yang membahas tarian atau seni pertunjukan tari dari Bali,12
7,Q8,pakaian adat,umum,Artikel yang topik utamanya pakaian/baju adat suatu suku atau daerah,7
8,Q9,upacara pernikahan adat,umum,Artikel yang topik utamanya tradisi atau prosesi pernikahan adat,5
9,Q10,senjata tradisional,umum,Artikel yang topik utamanya senjata tradisional (bukan tarian yang memakai senjata),11


### 7.2 Metrik

$$P@k = \frac{|\text{relevan} \cap \text{Top-}k|}{k} \qquad R@k = \frac{|\text{relevan} \cap \text{Top-}k|}{|\text{relevan}|} \qquad F1@k = \frac{2 \cdot P@k \cdot R@k}{P@k + R@k}$$

Catatan: jika jumlah dokumen relevan < k, nilai P@k maksimum hanya |relevan|/k (contoh: query *calung* punya 3 dokumen relevan, sehingga P@10 maksimum 0,3).

In [28]:
eval_df = evaluate(engine, queries, ks=(5, 10))
eval_df.to_csv("evaluation/hasil_evaluasi.csv", index=False, encoding="utf-8-sig")

per_query = eval_df.pivot(index="qid", columns="metode",
                          values=["P@5", "R@5", "F1@5", "P@10", "R@10", "F1@10"]).reindex(queries["qid"])
display(per_query.round(3))

P@5           R@5          F1@5        P@10          R@10         F1@10       
metode BM25 TF-IDF   BM25 TF-IDF   BM25 TF-IDF BM25 TF-IDF   BM25 TF-IDF   BM25 TF-IDF
qid                                                                                   
Q1      1.0    1.0  1.000  1.000  1.000  1.000  0.5    0.5  1.000  1.000  0.667  0.667
Q2      1.0    1.0  0.294  0.294  0.455  0.455  1.0    1.0  0.588  0.588  0.741  0.741
Q3      1.0    1.0  0.417  0.417  0.588  0.588  0.9    1.0  0.750  0.833  0.818  0.909
Q4      1.0    1.0  0.556  0.556  0.714  0.714  0.9    0.9  1.000  1.000  0.947  0.947
Q5      0.2    0.0  0.333  0.000  0.250  0.000  0.3    0.0  1.000  0.000  0.462  0.000
Q6      1.0    1.0  0.294  0.294  0.455  0.455  0.9    1.0  0.529  0.588  0.667  0.741
Q7      0.6    1.0  0.250  0.417  0.353  0.588  0.8    0.8  0.667  0.667  0.727  0.727
Q8      1.0    0.8  0.714  0.571  0.833  0.667  0.7    0.6  1.000  0.857  0.824  0.706
Q9      0.0    0.6  0.000  0.600  0.000  0.600  0.0    0.3  0.000  0.600  0.000  0.400
Q10     1.0    0.8  0.455  0.364  0.625  0.500  0.9    0.7  0.818  0.636  0.857  0.667
Q11     1.0    1.0  0.455  0.455  0.625  0.625  0.9    1.0  0.818  0.909  0.857  0.952
Q12     0.8    0.8  0.400  0.400  0.533  0.533  0.5    0.7  0.500  0.700  0.500  0.700
Q13     0.6    0.6  1.000  1.000  0.750  0.750  0.3    0.3  1.000  1.000  0.462  0.462
Q14     0.8    0.8  1.000  1.000  0.889  0.889  0.4    0.4  1.000  1.000  0.571  0.571
Q15     1.0    1.0  0.714  0.714  0.833  0.833  0.7    0.7  1.000  1.000  0.824  0.824
Q16     0.6    0.6  1.000  1.000  0.750  0.750  0.3    0.3  1.000  1.000  0.462  0.462
Q17     1.0    1.0  0.455  0.455  0.625  0.625  0.9    1.0  0.818  0.909  0.857  0.952
Q18     0.4    0.4  1.000  1.000  0.571  0.571  0.2    0.2  1.000  1.000  0.333  0.333

### 7.3 Rata-rata seluruh query dan per jenis query

In [29]:
print("Seluruh query (18):")
display(summarize(eval_df))
for jenis in ["umum", "pendek_spesifik"]:
    print(f"Jenis query: {jenis}")
    display(summarize(eval_df[eval_df["jenis"] == jenis]))

Seluruh query (18):


,P@5,R@5,F1@5,P@10,R@10,F1@10
metode,,,,,,
BM25,0.778,0.574,0.603,0.617,0.805,0.643
TF-IDF,0.800,0.585,0.619,0.633,0.794,0.653


Jenis query: umum


,P@5,R@5,F1@5,P@10,R@10,F1@10
metode,,,,,,
BM25,0.800,0.431,0.536,0.692,0.723,0.672
TF-IDF,0.833,0.447,0.560,0.708,0.698,0.680


Jenis query: pendek_spesifik


,P@5,R@5,F1@5,P@10,R@10,F1@10
metode,,,,,,
BM25,0.733,0.861,0.736,0.467,0.970,0.585
TF-IDF,0.733,0.861,0.736,0.483,0.985,0.601


## 8. Eksperimen Wajib

### Eksperimen 1 — ≥ 10 query, TF-IDF + VSM cosine dan BM25: Top-k dan skor

Top-3 setiap query untuk kedua metode beserta skornya (tanda ✓ = dokumen relevan menurut ground truth). Tabel lengkap Top-10 disimpan ke `evaluation/hasil_eksperimen1.csv`.

In [30]:
rows = []
for _, q in queries.iterrows():
    rel = set(q["relevant"])
    for m, name in [("tfidf", "TF-IDF"), ("bm25", "BM25")]:
        res = engine.search(q["query"], method=m, top_k=10)
        for r in res["results"]:
            rows.append({"qid": q["qid"], "query": q["query"], "metode": name, "peringkat": r["rank"],
                         "doc_id": r["doc_id"], "judul": r["title"], "skor": round(r["score"], 4),
                         "relevan": "✓" if r["doc_id"] in rel else ""})
exp1 = pd.DataFrame(rows)
exp1.to_csv("evaluation/hasil_eksperimen1.csv", index=False, encoding="utf-8-sig")

top3 = exp1[exp1["peringkat"] <= 3].copy()
top3["hasil"] = top3["doc_id"] + " " + top3["relevan"] + " (" + top3["skor"].astype(str) + ")"
display(top3.pivot_table(index=["qid", "query"], columns=["metode", "peringkat"],
                         values="hasil", aggfunc="first").reindex(queries["qid"], level=0))

metode                                        BM25                                               TF-IDF                                  
peringkat                                        1                 2                 3                1                2                3
qid query                                                                                                                                
Q1  ritual pemakaman suku toraja   D30 ✓ (17.8941)   D23 ✓ (17.5226)   D43 ✓ (17.2317)   D43 ✓ (0.3906)   D22 ✓ (0.3826)   D23 ✓ (0.3785)
Q2  rumah adat tradisional          D31 ✓ (5.4566)   D130 ✓ (5.4283)   D169 ✓ (5.3504)  D126 ✓ (0.3304)    D29 ✓ (0.301)   D65 ✓ (0.2955)
Q3  kain batik dan motifnya       D262 ✓ (12.4681)  D516 ✓ (12.3724)  D312 ✓ (12.1245)   D269 ✓ (0.688)  D273 ✓ (0.6469)  D262 ✓ (0.6081)
Q4  pertunjukan wayang             D267 ✓ (7.2275)   D305 ✓ (7.1534)   D429 ✓ (7.1048)  D348 ✓ (0.8198)  D373 ✓ (0.8039)  D305 ✓ (0.6514)
Q5  cerita rakyat papua            D208 ✓ (8.6394)    D463  (8.0849)    D356  (7.2447)    D12  (0.4327)   D139  (0.3139)   D144  (0.2206)
Q6  kain tenun tradisional         D129 ✓ (9.5375)      D66 ✓ (9.52)   D124 ✓ (9.5158)  D514 ✓ (0.7824)   D258 ✓ (0.685)  D124 ✓ (0.5832)
Q7  tari tradisional bali          D322 ✓ (7.3368)    D491  (7.2544)    D377  (6.8766)  D326 ✓ (0.3674)  D382 ✓ (0.3148)  D263 ✓ (0.2757)
Q8  pakaian adat                   D167 ✓ (4.1166)    D168 ✓ (4.103)   D127 ✓ (4.0185)  D168 ✓ (0.2944)  D167 ✓ (0.2741)   D62 ✓ (0.1646)
Q9  upacara pernikahan adat         D135  (6.8065)    D156  (6.1708)    D448  (5.7853)     D21  (0.269)  D136 ✓ (0.1887)   D45 ✓ (0.1746)
Q10 senjata tradisional             D49 ✓ (6.4816)   D116 ✓ (6.4188)   D477 ✓ (5.9708)  D116 ✓ (0.4518)   D49 ✓ (0.3737)  D350 ✓ (0.1455)
Q11 tari topeng                    D323 ✓ (7.4966)   D271 ✓ (7.4594)   D322 ✓ (7.3831)  D323 ✓ (0.8454)  D322 ✓ (0.8215)   D271 ✓ (0.779)
Q12 permainan tradisional anak     D408 ✓ (6.3557)   D407 ✓ (6.1278)   D101 ✓ (5.9325)   D408 ✓ (0.289)   D238  (0.2873)  D132 ✓ (0.2722)
Q13 angklung                       D274 ✓ (7.2469)   D289 ✓ (7.1175)   D403 ✓ (7.1012)  D274 ✓ (0.8066)  D403 ✓ (0.6725)   D289 ✓ (0.523)
Q14 reog                           D343 ✓ (8.1002)   D255 ✓ (8.0059)   D461 ✓ (7.7679)  D343 ✓ (0.6727)  D255 ✓ (0.5595)  D461 ✓ (0.4481)
Q15 gamelan                        D456 ✓ (4.6194)   D458 ✓ (4.6141)   D424 ✓ (4.5433)  D458 ✓ (0.7057)  D283 ✓ (0.6847)  D456 ✓ (0.5331)
Q16 calung                         D418 ✓ (8.2838)   D300 ✓ (8.2469)   D280 ✓ (7.9618)  D418 ✓ (0.8208)  D300 ✓ (0.8047)  D280 ✓ (0.6721)
Q17 gelung                          D13 ✓ (7.3423)    D12 ✓ (7.3239)      D8 ✓ (7.238)   D12 ✓ (0.5525)   D13 ✓ (0.4303)   D14 ✓ (0.4252)
Q18 ondel ondel                    D25 ✓ (18.1622)   D79 ✓ (17.7291)   D273  (14.7166)   D25 ✓ (0.9404)   D79 ✓ (0.8485)   D273  (0.1291)

### Eksperimen 2 — Query yang sama, VSM cosine vs BM25: perbedaan ranking

Untuk setiap query dihitung:
- **irisan Top-10**: berapa dokumen yang sama-sama muncul di Top-10 kedua metode,
- **perubahan Top-3**: dokumen yang masuk Top-3 di satu metode tetapi tidak di metode lain,
- **selisih F1@10** antara BM25 dan TF-IDF.

In [31]:
rows = []
for _, q in queries.iterrows():
    t10 = engine.rank_ids(q["query"], "tfidf", 10)
    b10 = engine.rank_ids(q["query"], "bm25", 10)
    f_t = eval_df[(eval_df.qid == q.qid) & (eval_df.metode == "TF-IDF")]["F1@10"].iloc[0]
    f_b = eval_df[(eval_df.qid == q.qid) & (eval_df.metode == "BM25")]["F1@10"].iloc[0]
    rows.append({"qid": q["qid"], "query": q["query"],
                 "irisan Top-10": len(set(t10) & set(b10)),
                 "Top-3 TF-IDF": ", ".join(t10[:3]), "Top-3 BM25": ", ".join(b10[:3]),
                 "Top-3 berubah": len(set(t10[:3]) ^ set(b10[:3])) // 2,
                 "F1@10 TF-IDF": round(f_t, 3), "F1@10 BM25": round(f_b, 3),
                 "selisih F1@10 (BM25 - TF-IDF)": round(f_b - f_t, 3)})
exp2 = pd.DataFrame(rows)
display(exp2)

,qid,query,irisan Top-10,Top-3 TF-IDF,Top-3 BM25,Top-3 berubah,F1@10 TF-IDF,F1@10 BM25,selisih F1@10 (BM25 - TF-IDF)
0,Q1,ritual pemakaman suku toraja,9,"D43, D22, D23","D30, D23, D43",1,0.667,0.667,0.000
1,Q2,rumah adat tradisional,5,"D126, D29, D65","D31, D130, D169",3,0.741,0.741,0.000
2,Q3,kain batik dan motifnya,9,"D269, D273, D262","D262, D516, D312",2,0.909,0.818,-0.091
3,Q4,pertunjukan wayang,9,"D348, D373, D305","D267, D305, D429",2,0.947,0.947,0.000
4,Q5,cerita rakyat papua,6,"D12, D139, D144","D208, D463, D356",3,0.000,0.462,0.462
5,Q6,kain tenun tradisional,8,"D514, D258, D124","D129, D66, D124",2,0.741,0.667,-0.074
6,Q7,tari tradisional bali,8,"D326, D382, D263","D322, D491, D377",3,0.727,0.727,0.000
7,Q8,pakaian adat,7,"D168, D167, D62","D167, D168, D127",1,0.706,0.824,0.118
8,Q9,upacara pernikahan adat,2,"D21, D136, D45","D135, D156, D448",3,0.400,0.000,-0.400
9,Q10,senjata tradisional,7,"D116, D49, D350","D49, D116, D477",1,0.667,0.857,0.190


**Query dengan perbedaan ranking terbesar** (irisan Top-10 paling kecil), dibandingkan berdampingan beserta panjang dokumen (jumlah token):

In [32]:
def bandingkan(qid):
    q = queries.set_index("qid").loc[qid]
    rel = set(q["relevant"])
    t = engine.tfidf.search(engine.pre.preprocess(q["query"]), 10)
    b = engine.bm25.search(engine.pre.preprocess(q["query"]), 10)
    fmt = lambda d, s: f"{'✓ ' if d in rel else '  '}{d} | {engine.docs.loc[d, 'title'][:45]} | {index.doc_len[d]} tok | {s:.3f}"
    print(f"{qid}: '{q['query']}'  ->  term {engine.pre.preprocess(q['query'])}")
    display(pd.DataFrame({"TF-IDF + cosine": [fmt(d, s) for d, s in t], "BM25": [fmt(d, s) for d, s in b]},
                         index=range(1, len(t) + 1)))

terbesar = exp2.sort_values(["irisan Top-10", "Top-3 berubah"], ascending=[True, False]).iloc[0]["qid"]
bandingkan(terbesar)

Q9: 'upacara pernikahan adat'  ->  term ['upacara', 'nikah', 'adat']


,TF-IDF + cosine,BM25
1,"D21 | Upacara Adat Beluluh, Simbol Harapan Orang Tu | 66 tok | 0.269","D135 | Mandi Kasai, Ritual Penyucian Diri di Lubukli | 326 tok | 6.806"
2,"✓ D136 | Pengadangan, Tradisi Unik Pernikahan Adat Suk | 150 tok | 0.189","D156 | Kuluk Beselang Mertuo, Penutup Kepala Penuh M | 434 tok | 6.171"
3,✓ D45 | Pesona Pernikahan Adat Lampung Melinting yang | 251 tok | 0.175,D448 | Simbolisme dan Kekuatan Tari Tandok dalam Bud | 412 tok | 5.785
4,D175 | Semarak Festival Erau Di Tenggarong | 285 tok | 0.150,"D33 | Mengenal Suku Saibatin, Pewaris Tradisi Pesis | 230 tok | 5.694"
5,✓ D69 | Menyaksikan Keunikan Pernikahan Adat Belitung | 205 tok | 0.109,D16 | Simpolong Tattong: Gaya Sanggul Bugis yang Me | 412 tok | 5.693
6,"D135 | Mandi Kasai, Ritual Penyucian Diri di Lubukli | 326 tok | 0.106","D491 | Rindik, Instrumen Bambu dengan Harmoni Khas B | 637 tok | 5.662"
7,"D284 | Upacara Bubuka, Simbol Keramahan Priangan dal | 182 tok | 0.106","D168 | Pakaian Adat Ternate, Cermin Kemegahan Kesult | 307 tok | 5.410"
8,D166 | Pakaian Adat Dayak Salako: Dari Bahan Alam hi | 352 tok | 0.106,"D122 | Sangkurat, Rompi Kebesaran Suku Dayak Ngaju | 176 tok | 5.377"
9,D18 | Menelusuri Tradisi Bali Aga di Desa Tenganan | 595 tok | 0.105,D6 | Falungku: Gelung Tradisional yang Mewakili Ke | 342 tok | 5.265
10,"D33 | Mengenal Suku Saibatin, Pewaris Tradisi Pesis | 230 tok | 0.099",D20 | Yadnya Kasada: Ritual Adat Suku Tengger di Gu | 614 tok | 5.122


In [33]:
selisih_f1 = exp2.loc[exp2["selisih F1@10 (BM25 - TF-IDF)"].abs().idxmax(), "qid"]
print("Query dengan selisih F1@10 terbesar:", selisih_f1)
bandingkan(selisih_f1)

Query dengan selisih F1@10 terbesar: Q5
Q5: 'cerita rakyat papua'  ->  term ['cerita', 'rakyat', 'papua']


,TF-IDF + cosine,BM25
1,D12 | Gelung Papua: Simbol Keanggunan dan Kreativit | 366 tok | 0.433,✓ D208 | Cerita Rakyat Papua: Biwar Sang Penakluk Naga | 651 tok | 8.639
2,D139 | Kecantikan Wanita dalam Anyaman Rambut Khas P | 226 tok | 0.314,"D463 | Tifa Papua, Nada Ritmik Khas Bumi Papua | 583 tok | 8.085"
3,"D144 | Mengenal Pinang, ‘Permen’ Masyarakat Papua | 361 tok | 0.221","D356 | Tari Yospan, Tarian Persahabatan Rakyat Papua | 192 tok | 7.245"
4,"D356 | Tari Yospan, Tarian Persahabatan Rakyat Papua | 192 tok | 0.190",D12 | Gelung Papua: Simbol Keanggunan dan Kreativit | 366 tok | 7.028
5,"D463 | Tifa Papua, Nada Ritmik Khas Bumi Papua | 583 tok | 0.178","D474 | Mengenal Atawenani, Lagu Pengiring Tari Wutuk | 291 tok | 6.933"
6,"D261 | Wisata Festival Bahari, di Raja Ampat | 210 tok | 0.158",✓ D231 | Cerita Rakyat Papua: Empat Raja | 749 tok | 6.760
7,D478 | Memanggil dengan Merdu Lewat Alat Musik Fuu | 471 tok | 0.153,"D467 | Tari Wutukala, Ekspresi Syukur Nelayan Papua | 306 tok | 5.930"
8,"D474 | Mengenal Atawenani, Lagu Pengiring Tari Wutuk | 291 tok | 0.133",D319 | Seni Patung Asmat: Jenis dan Makna Leluhur di | 348 tok | 5.686
9,"D152 | Rumah Kaki Seribu, Arsitektur Mengagumkan dar | 581 tok | 0.106",✓ D178 | Cerita Rakyat Papua: Legenda Cabo Pui dan Bat | 1270 tok | 5.573
10,"D467 | Tari Wutukala, Ekspresi Syukur Nelayan Papua | 306 tok | 0.094",D139 | Kecantikan Wanita dalam Anyaman Rambut Khas P | 226 tok | 5.477


### Eksperimen 3 — Query pendek & spesifik: pengaruh term dan panjang dokumen

Query satu term (Q13–Q18). Untuk setiap dokumen di Top-5 ditampilkan TF term query dan panjang dokumen, agar terlihat bagaimana kedua metode memperlakukan TF dan panjang dokumen.

In [34]:
for qid in ["Q13", "Q15", "Q16"]:
    q = queries.set_index("qid").loc[qid]
    terms = engine.pre.preprocess(q["query"])
    rel = set(q["relevant"])
    rows = []
    for name, retr in [("TF-IDF", engine.tfidf), ("BM25", engine.bm25)]:
        for rank, (d, s) in enumerate(retr.search(terms, 5), start=1):
            rows.append({"metode": name, "peringkat": rank, "doc_id": d,
                         "relevan": "✓" if d in rel else "",
                         "TF": sum(index.tf(t, d) for t in terms), "|D| (token)": index.doc_len[d],
                         "TF / |D| (%)": round(100 * sum(index.tf(t, d) for t in terms) / index.doc_len[d], 2),
                         "skor": round(s, 4), "judul": engine.docs.loc[d, "title"][:55]})
    print(f"{qid}: '{q['query']}'  (DF = {index.df(terms[0])}, avgdl = {index.avgdl:.0f})")
    display(pd.DataFrame(rows))

Q13: 'angklung'  (DF = 16, avgdl = 376)


,metode,peringkat,doc_id,relevan,TF,|D| (token),TF / |D| (%),skor,judul
0,TF-IDF,1,D274,✓,14,142,9.86,0.8066,Angklung: Dari Tradisi Agraris Jawa Barat hingga Warisa
1,TF-IDF,2,D403,✓,28,660,4.24,0.6725,Angklung: Warisan Budaya Sunda yang Diakui UNESCO
2,TF-IDF,3,D289,✓,10,145,6.90,0.5230,"Angklung Buhun, Pengiring Ritual dan Identitas Masyarak"
3,TF-IDF,4,D280,,3,195,1.54,0.1776,Calung: Merdunya Ketukan Bambu dari Tanah Sunda
4,TF-IDF,5,D300,,3,188,1.60,0.1119,"Calung Renteng, Wasiat Kebudayaan Agraris"
5,BM25,1,D274,✓,14,142,9.86,7.2469,Angklung: Dari Tradisi Agraris Jawa Barat hingga Warisa
6,BM25,2,D289,✓,10,145,6.90,7.1175,"Angklung Buhun, Pengiring Ritual dan Identitas Masyarak"
7,BM25,3,D403,✓,28,660,4.24,7.1012,Angklung: Warisan Budaya Sunda yang Diakui UNESCO
8,BM25,4,D300,,3,188,1.60,6.0620,"Calung Renteng, Wasiat Kebudayaan Agraris"
9,BM25,5,D280,,3,195,1.54,6.0350,Calung: Merdunya Ketukan Bambu dari Tanah Sunda


Q15: 'gamelan'  (DF = 59, avgdl = 376)


,metode,peringkat,doc_id,relevan,TF,|D| (token),TF / |D| (%),skor,judul
0,TF-IDF,1,D458,✓,52,545,9.54,0.7057,"Gamelan Bali, Ragam Keindahan Selaras Zaman"
1,TF-IDF,2,D283,✓,13,134,9.70,0.6847,"Gamelan, Kesenian Adiluhung dari Jawa"
2,TF-IDF,3,D456,✓,54,544,9.93,0.5331,Jenis-Jenis Gamelan Sunda dan Asal-usulnya
3,TF-IDF,4,D424,✓,19,247,7.69,0.4791,Menulusuri Tahapan Rumit dalam Pembuatan Gamelan
4,TF-IDF,5,D428,✓,11,260,4.23,0.4622,"Menelusuri Gamelan Sunda: Ciri Khas, Nilai, dan Pelesta"
5,BM25,1,D456,✓,54,544,9.93,4.6194,Jenis-Jenis Gamelan Sunda dan Asal-usulnya
6,BM25,2,D458,✓,52,545,9.54,4.6141,"Gamelan Bali, Ragam Keindahan Selaras Zaman"
7,BM25,3,D424,✓,19,247,7.69,4.5433,Menulusuri Tahapan Rumit dalam Pembuatan Gamelan
8,BM25,4,D283,✓,13,134,9.70,4.5397,"Gamelan, Kesenian Adiluhung dari Jawa"
9,BM25,5,D428,✓,11,260,4.23,4.3884,"Menelusuri Gamelan Sunda: Ciri Khas, Nilai, dan Pelesta"


Q16: 'calung'  (DF = 10, avgdl = 376)


,metode,peringkat,doc_id,relevan,TF,|D| (token),TF / |D| (%),skor,judul
0,TF-IDF,1,D418,✓,22,195,11.28,0.8208,"Sejarah dan Ragam Calung, Ikon Budaya Sunda"
1,TF-IDF,2,D300,✓,19,188,10.11,0.8047,"Calung Renteng, Wasiat Kebudayaan Agraris"
2,TF-IDF,3,D280,✓,10,195,5.13,0.6721,Calung: Merdunya Ketukan Bambu dari Tanah Sunda
3,TF-IDF,4,D375,,2,283,0.71,0.0656,"Tari Rong Tek, Lengger Banyumasan dalam Wajah Baru"
4,TF-IDF,5,D274,,1,142,0.70,0.0654,Angklung: Dari Tradisi Agraris Jawa Barat hingga Warisa
5,BM25,1,D418,✓,22,195,11.28,8.2838,"Sejarah dan Ragam Calung, Ikon Budaya Sunda"
6,BM25,2,D300,✓,19,188,10.11,8.2469,"Calung Renteng, Wasiat Kebudayaan Agraris"
7,BM25,3,D280,✓,10,195,5.13,7.9618,Calung: Merdunya Ketukan Bambu dari Tanah Sunda
8,BM25,4,D375,,2,283,0.71,5.7573,"Tari Rong Tek, Lengger Banyumasan dalam Wajah Baru"
9,BM25,5,D274,,1,142,0.70,5.2266,Angklung: Dari Tradisi Agraris Jawa Barat hingga Warisa


In [35]:
print("Ringkasan query pendek & spesifik:")
display(eval_df[eval_df["jenis"] == "pendek_spesifik"]
        .pivot(index="qid", columns="metode", values=["P@5", "R@5", "F1@5", "R@10"]).round(3))

Ringkasan query pendek & spesifik:


P@5           R@5          F1@5          R@10       
metode BM25 TF-IDF   BM25 TF-IDF   BM25 TF-IDF   BM25 TF-IDF
qid                                                         
Q13     0.6    0.6  1.000  1.000  0.750  0.750  1.000  1.000
Q14     0.8    0.8  1.000  1.000  0.889  0.889  1.000  1.000
Q15     1.0    1.0  0.714  0.714  0.833  0.833  1.000  1.000
Q16     0.6    0.6  1.000  1.000  0.750  0.750  1.000  1.000
Q17     1.0    1.0  0.455  0.455  0.625  0.625  0.818  0.909
Q18     0.4    0.4  1.000  1.000  0.571  0.571  1.000  1.000

## 9. Pembanding: scikit-learn `TfidfVectorizer`

Sebagai pembanding (sesuai saran materi kuliah), ranking TF-IDF manual dibandingkan dengan `TfidfVectorizer` + `cosine_similarity` dari scikit-learn, memakai token hasil prapemrosesan yang sama.
Rumus IDF scikit-learn berbeda ($\ln\frac{1+N}{1+df} + 1$), sehingga skor tidak identik; yang dibandingkan adalah kemiripan Top-10.

In [36]:
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    vectorizer = TfidfVectorizer(analyzer=lambda tokens: tokens)
    X = vectorizer.fit_transform(df["tokens"])
    doc_ids = df["doc_id"].tolist()

    rows = []
    for _, q in queries.iterrows():
        terms = engine.pre.preprocess(q["query"])
        sims = cosine_similarity(vectorizer.transform([terms]), X).ravel()
        sk_top = [doc_ids[i] for i in sims.argsort()[::-1][:10] if sims[i] > 0]
        manual_top = engine.rank_ids(q["query"], "tfidf", 10)
        rows.append({"qid": q["qid"], "query": q["query"],
                     "irisan Top-10 (manual vs sklearn)": len(set(sk_top) & set(manual_top)),
                     "peringkat 1 sama": sk_top[:1] == manual_top[:1]})
    sk = pd.DataFrame(rows)
    display(sk)
    print("Rata-rata irisan Top-10:", round(sk.iloc[:, 2].mean(), 2), "dari 10")
except ImportError:
    print("scikit-learn belum terpasang (pip install scikit-learn); bagian pembanding dilewati.")

,qid,query,irisan Top-10 (manual vs sklearn),peringkat 1 sama
0,Q1,ritual pemakaman suku toraja,10,True
1,Q2,rumah adat tradisional,9,True
2,Q3,kain batik dan motifnya,10,True
3,Q4,pertunjukan wayang,9,True
4,Q5,cerita rakyat papua,8,True
5,Q6,kain tenun tradisional,10,True
6,Q7,tari tradisional bali,7,True
7,Q8,pakaian adat,9,False
8,Q9,upacara pernikahan adat,8,True
9,Q10,senjata tradisional,9,True


Rata-rata irisan Top-10: 9.11 dari 10


## 10. Analisis dan Kesimpulan

### 10.1 Hasil keseluruhan

| Metode | P@5 | R@5 | F1@5 | P@10 | R@10 | F1@10 |
|---|---|---|---|---|---|---|
| TF-IDF + cosine | 0,800 | 0,585 | 0,619 | 0,633 | 0,794 | 0,653 |
| BM25 | 0,778 | 0,574 | 0,603 | 0,617 | 0,805 | 0,643 |

Rata-rata kedua metode **hampir sama** (selisih F1@10 hanya 0,010). Pada 9 dari 18 query nilai F1@10 keduanya identik; BM25 lebih baik pada 3 query (Q5, Q8, Q10) dan TF-IDF + cosine lebih baik pada 6 query (Q3, Q6, Q9, Q11, Q12, Q17), umumnya dengan selisih kecil. Pada koleksi ini tidak ada metode yang selalu lebih baik — hasilnya bergantung pada karakteristik query.

### 10.2 Mengapa ranking berbeda — query dengan perbedaan terbesar

**Q9 *upacara pernikahan adat*** (irisan Top-10 hanya 2 dokumen; F1@10 TF-IDF 0,400 vs BM25 0,000).
Dokumen relevan (D136, D45, D69, D34) **tidak memuat kata *upacara*** — mereka menulis *pernikahan adat*, *prosesi*, atau *pengantin*. BM25 menjumlahkan kontribusi setiap term, dan karena TF saturation setiap term hanya menyumbang ±2–3 poin. Akibatnya dokumen yang memuat **ketiga** term walaupun hanya sekilas (D135 *Mandi Kasai*: upacara 4×, nikah 9×, adat 4× → 6,81) mengalahkan dokumen relevan yang sangat fokus pada *nikah* tetapi tanpa *upacara* (D45: nikah 12×, adat 6× → 4,82). Cosine similarity justru mengukur **proporsi**: D136 (150 token, nikah 10×) dan D45 mendapat skor cosine tinggi karena vektornya didominasi term query, sehingga naik ke peringkat 2–3. Ini contoh **vocabulary mismatch**: kata di query tidak dipakai oleh dokumen relevan.

**Q5 *cerita rakyat papua*** (F1@10 TF-IDF 0,000 vs BM25 0,462) — kebalikan dari Q9.
TF-IDF memakai TF mentah, sehingga term langka *papua* (DF 36, IDF tinggi) yang muncul berkali-kali mendominasi vektor dokumen. Artikel seperti *Gelung Papua* atau *Anyaman Rambut Khas Papua* menang walaupun tidak memuat *cerita rakyat*. BM25 membatasi pengaruh pengulangan *papua* lewat **TF saturation**, sehingga dokumen yang memuat ketiga term (D208 *Cerita Rakyat Papua: Biwar*) naik ke peringkat 1.

**Q10 *senjata tradisional*** (F1@10 BM25 0,857 vs TF-IDF 0,667) — TF-IDF menaikkan tarian yang banyak menyebut *senjata* (D264 *Tari Kabasaran*, D455 *Tari Perang Suku Dani*), sedangkan BM25 lebih menghargai dokumen yang memuat kedua term *senjata* dan *tradisional*, yaitu artikel senjata tradisional itu sendiri.

**Q12 *permainan tradisional anak*** (F1@10 TF-IDF 0,700 vs BM25 0,500) — stemming menyatukan *permainan* dan *dimainkan* menjadi term yang sama (*main*). Artikel alat musik (D418/D300 *Calung*, D490 *Salude*) banyak menulis "alat musik ini *dimainkan*…", sehingga memuat ketiga term query dan dinaikkan BM25. Cosine lebih memilih artikel yang memang didominasi kata *main(an)*. Ini contoh efek samping stemming: dua makna berbeda menjadi satu term.

### 10.3 Eksperimen 3 — query pendek & spesifik

Untuk query satu term (*angklung, reog, gamelan, calung, gelung, ondel ondel*) kedua metode menghasilkan Top-10 yang hampir sama (irisan 7–10 dokumen) dan **F1@5 identik** (0,736). Dengan satu term, skor hanya ditentukan oleh TF term itu dan panjang dokumen, sehingga dokumen relevan hampir selalu berada di atas. Perbedaan terlihat pada **urutan di antara dokumen relevan**:
- *angklung*: TF-IDF menaruh D403 (TF 28, 660 token) di peringkat 2, sedangkan BM25 menaruh D289 (TF 10, 145 token) di peringkat 2. BM25 menurunkan D403 karena dokumennya **1,8× lebih panjang dari rata-rata** dan TF 28 sudah jenuh (saturasi), sementara dokumen pendek dengan TF 10 sudah dianggap cukup relevan.
- *gamelan*: TF-IDF menaruh D283 (13×, 134 token) di peringkat 2 karena proporsinya tinggi, BM25 menaruhnya di peringkat 4 karena TF absolutnya kecil dibanding D456/D458 (52–54×).

P@10 query pendek rendah (±0,47) bukan karena metode gagal, tetapi karena **jumlah dokumen relevan sedikit** (2–11 dokumen), sehingga P@10 maksimum memang kecil; recall@10 hampir sempurna (0,97–0,99).

### 10.4 Pengaruh TF, IDF, dan panjang dokumen (rangkuman Bagian 5.4)

- **TF**: pada TF-IDF bobot naik linear (TF 50 → bobot 50×), pada BM25 naik makin kecil dan mendekati batas k1 + 1 = 2,2 (TF 1 → 1,00; TF 10 → 1,96; TF 50 → 2,15). Pengulangan term tidak meningkatkan skor secara linear.
- **IDF**: term langka sangat menentukan skor — *sasando* (DF 1) mempunyai IDF 2,71 (TF-IDF) / 5,84 (BM25), sedangkan *budaya* (DF 333) hanya 0,19 / 0,44.
- **Panjang dokumen**: dengan b = 0,75 skor term ber-TF 3 turun dari 1,87 (dokumen ¼ avgdl) menjadi 0,96 (dokumen 4× avgdl); dengan b = 0 panjang dokumen tidak berpengaruh. Pada TF-IDF, panjang dokumen dinormalisasi lewat panjang vektor di cosine.

### 10.5 Kesimpulan

1. Sistem mesin pencari berhasil dibangun lengkap: prapemrosesan → inverted index (11.639 term, 516 dokumen) → TF-IDF/VSM + cosine dan BM25 → Top-k dengan judul, URL, skor, dan snippet → antarmuka CLI dan Streamlit.
2. Implementasi diverifikasi dengan contoh materi kuliah (TF-IDF *wifi* = 3,0; BM25 D1 = 0,9400) dan dengan scikit-learn (irisan Top-10 rata-rata 9,1 dari 10).
3. TF-IDF + cosine dan BM25 menghasilkan kualitas rata-rata yang setara pada koleksi ini (F1@10 0,653 vs 0,643). BM25 unggul ketika satu term diulang-ulang oleh dokumen yang tidak relevan (Q5, Q10); TF-IDF + cosine unggul ketika dokumen relevan tidak memuat semua term query atau ketika term query ambigu (Q9, Q12).
4. Prapemrosesan berpengaruh besar: stemming menyatukan *tari/tarian/menari/penari*, sedangkan penanganan nama diri dan koreksi stemming mencegah term salah seperti *bali → bal* atau *menikah → meni*.

### 10.6 Keterbatasan dan pengembangan

- Kedua metode bersifat **lexical**: tidak mengenali sinonim (*pernikahan* vs *perkawinan*, *upacara* vs *prosesi*) dan tidak membedakan makna kata yang di-stem sama (*permainan* vs *dimainkan*). Pengembangan: query expansion dengan tesaurus atau semantic retrieval (embedding) pada Tugas 3.
- Ground truth disusun manual oleh satu penilai (judul + pooling Top-10 kedua metode). Dokumen relevan yang tidak masuk Top-10 metode mana pun dan judulnya tidak menyebut kata kunci mungkin masih terlewat, sehingga recall yang dilaporkan bisa sedikit lebih tinggi dari kenyataan.
- Parameter BM25 (k1 = 1,2; b = 0,75) belum di-tuning; nilai lain dapat dicoba dengan data evaluasi yang lebih besar.